# XGBoost

In [2]:
import pandas as pd
import numpy as np
import re
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_validate
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, make_scorer, classification_report, roc_auc_score
from tabulate import tabulate

from pathlib import Path
import warnings

import re



import time
from datetime import timedelta

# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')


"""
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'

"""


# Lista dei csv su cui fare training
datasets = {

    'dataset_fuso': FILE_PATH / 'dataset_fuso.csv'
}

# Training


In [3]:
def training(file_path, csv_name):
   
    df = pd.read_csv(file_path)

    
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]', 'HER2 [SII]']

    
    df_validi = df.dropna(subset=original_target_list).copy()

    
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    df_validi['HER2_class'] = (df_validi['HER2 [SII]'] >= 3).astype(int)

    
    final_target_list = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

    
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # Separo features e target
    target = df_validi[final_target_list]
    # Mi salvo gli ID dei pazienti per dopo
    groups = df_validi['Patient ID']

   
    features = features.fillna(features.mean())
   
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]

    
    cv = GroupKFold(n_splits=5)

    # Definisco i parametri base per XGBoost
    base_model = XGBClassifier(
        random_state=42,
        n_jobs=1,                # 1 core per modello, parallelizzo dopo la griglia
        subsample=0.8,           # Regolarizzazione "gratis"
        colsample_bytree=0.8,    # Altra regolarizzazione
        eval_metric='logloss'    # Zittisco i warning noiosi
    )
    # Wrappo tutto per gestire 4 output insieme.
    multi_output_model = MultiOutputClassifier(base_model)

    # Definisco gli iperparametri
    iperparametri = {
      'estimator__n_estimators': [50, 100, 150],      
      'estimator__max_depth': [2, 3],                 # Se imposto profonditá alta, il modello impara a memoria
      'estimator__learning_rate': [0.05, 0.1],       
      'estimator__min_child_weight': [2, 4]           
    }

    
    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            # Macro avg gestisce bene le classi sbilanciate
            s = f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0)
            scores.append(s)
        return np.mean(scores)

    scorer = make_scorer(multi_f1_scorer)

    print(f"\nInizio Grid Search per: {csv_name}")

    # Addestro
    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=cv,                  
        scoring=scorer,
        n_jobs=-1,              
        verbose=1,
        return_train_score=False 
    )

    
    grid_search.fit(features, target, groups=groups)

    
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_

    fold_reports = []

    
    clean_best_params = {k.replace('estimator__', ''): v for k, v in best_params.items()}

   
    final_model_params = {
        'random_state': 42,
        'n_jobs': 1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'eval_metric': 'logloss',
        **clean_best_params
    }

    
    for train_idx, test_idx in cv.split(features, target, groups):
        # Divido i dati esattamente come ha fatto la GridSearch
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        # Creo un clone del modello vincente
        model_clone = MultiOutputClassifier(XGBClassifier(**final_model_params))
        model_clone.fit(X_train, y_train)

        # Predico e genero il report dettagliato
        y_pred = model_clone.predict(X_test)
        
        # Ottengo le probabilità per calcolare la AUC
        y_proba_list = model_clone.predict_proba(X_test)

        report_dict = {}
        for i, col in enumerate(final_target_list):
            # Salvo Precision, Recall, F1 per ogni singolo target (PR, ER, ecc.)
            rep = classification_report(
                y_test.iloc[:, i],
                y_pred[:, i],
                output_dict=True,
                zero_division=0
            )
            # Verifico quante classi uniche ci sono nel test set reale
            unique_classes = np.unique(y_test.iloc[:, i])

            if len(unique_classes) < 2:
                # Impossibile calcolare AUC se c'è solo una classe nel ground truth
                auc_val = np.nan 
            else:
                try:
                    # Controllo se il modello ha prodotto probabilità per la classe positiva
                    if y_proba_list[i].shape[1] == 2:
                        auc_val = roc_auc_score(y_test.iloc[:, i], y_proba_list[i][:, 1])
                    else:
                        # Il modello ha predetto solo una classe 
                        auc_val = 0.5 
                except ValueError:
                    auc_val = np.nan
            
            # Inseriamo la AUC nel dizionario del report
            rep['auc'] = auc_val
            report_dict[col] = rep

        fold_reports.append(report_dict)

    # Impacchetto tutto il malloppo in un formato che la funzione di stampa capisce.
    final_result = [{
        **clean_best_params,
     
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'mean_score': best_score,
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports
    }]

    return final_result


# Vado a stampare il risultato in un formato leggibile

In [4]:
def print_grid_search_results(results_per_dataset):

    print("\n" + "=" * 80)
    print(" " * 25 + "RIEPILOGO DEI MIGLIORI RISULTATI (XGBoost)")
    print("=" * 80)

    summary_data = []

    for name, metrics_list in results_per_dataset.items():
        best_result = metrics_list[0]

        # Calcolo della AUC media
        auc_values = []
        if best_result.get('fold_reports'):
            for fold_rep in best_result['fold_reports']:
                for target_key, target_metrics in fold_rep.items():
                    if isinstance(target_metrics, dict) and 'auc' in target_metrics:
                        auc_values.append(target_metrics['auc'])

        # Uso nanmean per ignorare i casi in cui AUC non era calcolabile
        mean_auc = np.nanmean(auc_values) if auc_values else 0.0


        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}")
        print(f" Mean AUC    = {mean_auc:.3f}\n")

        print("Iperparametri Ottimali:")

        params_to_show = [
            ('n_estimators', 'n_estimators'),
            ('max_depth', 'max_depth'),
            ('learning_rate', 'learning_rate'),
            ('min_child_weight', 'min_child_weight'),
            ('subsample', 'subsample'),             
            ('colsample_bytree', 'colsample_bytree') 
        ]

        params_table = []
        for label, key in params_to_show:
            val = best_result.get(key)
            if val is not None:
                params_table.append([label, val])

        print(tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))

        print("\n Metriche di Classificazione per Target (Dettaglio):\n")
        target_names = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

        if best_result['fold_reports']:
            first_fold_report = best_result['fold_reports'][0]

            for target_name in target_names:
                if target_name not in first_fold_report:
                    continue

                current_target_report = first_fold_report[target_name]

                rows = []
                classes = [c for c in ['0', '1'] if c in current_target_report]

                for cls in classes:
                    rows.append([
                        f"Classe {cls}",
                        f"{current_target_report[cls]['precision']:.3f}",
                        f"{current_target_report[cls]['recall']:.3f}",
                        f"{current_target_report[cls]['f1-score']:.3f}",
                        int(current_target_report[cls]['support'])
                    ])

                # AUC individuale
                auc_val = current_target_report.get('auc')
                if auc_val is not None and not np.isnan(auc_val):
                    auc_str = f"  ---> AUC: {auc_val:.3f}"
                else:
                    auc_str = "  ---> AUC: N/A"

                print(f"  Target: {target_name} {auc_str}")
                print(tabulate(rows, headers=['', 'Precision', 'Recall', 'F1-score', 'Support'],
                             tablefmt='simple', colalign=('left', 'center', 'center', 'center', 'center')))
                print()
        else:
            print("Nessun report dettagliato disponibile.")

        summary_data.append([
            name,
            f"{best_result['mean_score']:.3f}",
            f"{best_result['std_score']:.3f}",
            f"{mean_auc:.3f}", 
            best_result.get('max_depth'),
            best_result.get('learning_rate'),
            best_result.get('n_estimators')
        ])

    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO TRA TUTTI I DATASET")
    print("=" * 80 + "\n")

    summary_data.sort(key=lambda x: float(x[1]), reverse=True)

    # Aggiornati header per includere AUC e nomi corretti
    print(tabulate(summary_data,
                   headers=['Dataset', 'F1-score', 'Std Dev', 'AUC', 'Max Depth', 'LR', 'N Est.'],
                   tablefmt='grid',
                   floatfmt=('.3f', '.3f', '.3f', '.3f', '.0f', '.0f', '.0f')))

# Lettura dei file

In [5]:
start_time = time.time()


# Eseguo il training per tutti i dataset
results_per_dataset = {}
for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Usa la nuova funzione per stampare i risultati
print_grid_search_results(results_per_dataset)




end_time = time.time()
# Calcolo il tempo impiegato
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



Inizio Grid Search per: dataset_fuso
Fitting 5 folds for each of 24 candidates, totalling 120 fits

                         RIEPILOGO DEI MIGLIORI RISULTATI (XGBoost)

────────────────────────────────────────────────────────────────────────────────
 Dataset: dataset_fuso
────────────────────────────────────────────────────────────────────────────────

 Performance: F1-score = 0.672 ± 0.045
 Mean AUC    = 0.643

Iperparametri Ottimali:
Parametro           Valore
----------------  --------
n_estimators         150
max_depth              2
learning_rate          0.1
min_child_weight       2
subsample              0.8
colsample_bytree       0.8

 Metriche di Classificazione per Target (Dettaglio):

  Target: PR_class   ---> AUC: 0.667
           Precision    Recall    F1-score    Support
--------  -----------  --------  ----------  ---------
Classe 0     0.25       0.667      0.364         3
Classe 1     0.75       0.333      0.462         9

  Target: ER_class   ---> AUC: 0.909
        